# 🌌 Unified Adaptive Low-Light Enhancement & Object Detection Pipeline
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/samba18071-bot/low_light_enhancement_detection_bundle/blob/main/unified_adaptive_low_light_pipeline.ipynb)

### Self-Contained End-to-End System with Interactive Upload, YOLO 11n, Sliced Tiling (SAHI) & Micro-Particle Saliency

This notebook unifies the complete computer vision pipeline into a single, standalone interactive environment:

1. **Continuous Content-Adaptive Illumination (Approach B)**:
   - High-pass residual Median Absolute Deviation (MAD) sensor noise estimation ($\sigma$).
   - Dynamic 256-entry Gamma LUT ($O(1)$ contrast expansion in LAB space).
   - Illuminated edge-preserving filtering (Bilateral & Fast Non-Local Means).
   - Controlled CLAHE and noise-bounded high-frequency detail sharpening.
2. **Photorealistic Generative Night-to-Day Style Harmonization**:
   - Deep tone curves, daylight color cast transfer, and shadow detail recovery.
3. **Deep Learning Object Detection (YOLO 11n)**:
   - Pure OpenCV DNN CPU execution with fallback to Ultralytics PyTorch backend.
   - Sliced Aided Hyper Inference (SAHI) with overlapping high-resolution tiles for minute/distant targets.
   - Morphological Top-Hat micro-particle and point-light saliency detection down to 1–2 pixels.
4. **Temporal Video Stabilization & Multi-Object Tracking**:
   - EMA parameter smoothing, scene-cut detection, and continuous IoU trajectory association.
5. **Interactive Upload & Processing Hub**:
   - Integrated `ipywidgets` upload interface supporting images and videos with instant side-by-side comparative visualization.


## 1. Environment Setup, Imports & Model Discovery
Initializes core dependencies (OpenCV, NumPy, Matplotlib, IPyWidgets) and auto-resolves the YOLO 11n model weights.


In [ ]:
import os
import sys
import time
import math
import json
import tempfile
import glob
from typing import List, Dict, Any, Tuple, Optional, Callable

# Check if running in Google Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    if not os.path.exists('models/yolo11n.onnx'):
        if not os.path.exists('low_light_enhancement_detection_bundle'):
            os.system('git clone https://github.com/samba18071-bot/low_light_enhancement_detection_bundle.git')
        if os.path.exists('low_light_enhancement_detection_bundle'):
            os.chdir('low_light_enhancement_detection_bundle')
        sys.path.insert(0, os.getcwd())

import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

# Auto-locate YOLO 11n ONNX weights
POSSIBLE_MODEL_PATHS = [
    "models/yolo11n.onnx",
    "../models/yolo11n.onnx",
    os.path.join(os.getcwd(), "models", "yolo11n.onnx"),
    "yolo11n.onnx"
]

RESOLVED_MODEL_PATH = None
for p in POSSIBLE_MODEL_PATHS:
    if os.path.exists(p):
        RESOLVED_MODEL_PATH = os.path.abspath(p)
        break

print("=" * 65)
print("🚀 UNIFIED ADAPTIVE LOW-LIGHT SYSTEM INITIALIZED")
print("=" * 65)
print(f"• OpenCV Version:    {cv2.__version__}")
print(f"• NumPy Version:     {np.__version__}")
print(f"• Resolved Model:    {RESOLVED_MODEL_PATH}")
print("=" * 65)


## 2. Low-Light Illumination Recovery & Continuous Adaptive Engine
Implements the continuous mathematical formulations for sensor noise estimation, dynamic gamma calculation, illuminated edge-preserving denoising, CLAHE, and temporal smoothing.


In [ ]:
# ------------------------------------------------------------------------------
# 2.1 Noise & Luminance Estimator
# ------------------------------------------------------------------------------
def estimate_noise_sigma(gray: np.ndarray) -> float:
    """Estimates Gaussian sensor noise standard deviation via high-pass MAD."""
    gray_f = gray.astype(np.float32)
    smooth = cv2.GaussianBlur(gray_f, (0, 0), sigmaX=1.0, sigmaY=1.0)
    residual = gray_f - smooth
    med = np.median(residual)
    mad = np.median(np.abs(residual - med))
    return float(mad / 0.6745)


def select_adaptive_parameters(frame_bgr: np.ndarray) -> Dict[str, Any]:
    """Computes content-adaptive illumination parameters from frame statistics."""
    lab = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2LAB)
    l_channel = lab[:, :, 0]
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)

    p10 = float(np.percentile(l_channel, 10))
    p50 = float(np.percentile(l_channel, 50))
    p90 = float(np.percentile(l_channel, 90))
    sigma = estimate_noise_sigma(gray)

    # Normalized darkness indicator [0.0 = bright, 1.0 = pitch dark]
    darkness = float(np.clip((128.0 - p50) / 128.0, 0.0, 1.0))

    # Dynamic gamma curve in [0.45, 1.0]
    gamma = float(np.clip(1.0 - 0.55 * darkness, 0.45, 1.0))

    # Bilateral / NLMeans filter strength proportional to estimated noise
    denoise_h = float(np.clip(0.8 * sigma + 3.0 * darkness, 3.0, 25.0))

    # Adaptive CLAHE clip limit
    clahe_clip = float(np.clip(1.2 + 1.8 * darkness - 0.05 * sigma, 1.0, 3.0))

    # Noise-bounded high-frequency unsharp masking
    sharpen = float(np.clip(0.12 - 0.005 * sigma, 0.02, 0.15))

    return {
        "p10": p10, "p50": p50, "p90": p90,
        "noise_sigma": sigma,
        "darkness": darkness,
        "gamma": gamma,
        "denoise_h": denoise_h,
        "clahe_clip": clahe_clip,
        "clahe_mix": 0.45,
        "sharpen": sharpen,
        "sat_boost": 1.15
    }


# ------------------------------------------------------------------------------
# 2.2 Continuous Adaptive Enhancer
# ------------------------------------------------------------------------------
class AdaptiveEnhancer:
    """Continuous Content-Adaptive Illumination & Illuminated Denoising Engine."""
    def __init__(self, denoise_method: str = "bilateral"):
        self.denoise_method = denoise_method.lower()

    @staticmethod
    def _build_gamma_lut(gamma: float) -> np.ndarray:
        inv_gamma = 1.0 / max(gamma, 0.01)
        table = np.array([((i / 255.0) ** inv_gamma) * 255.0 for i in range(256)]).astype(np.uint8)
        return table

    def enhance(self, frame_bgr: np.ndarray, params: Optional[Dict[str, Any]] = None) -> np.ndarray:
        if params is None:
            params = select_adaptive_parameters(frame_bgr)

        lab = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2LAB)
        l_orig, a_chan, b_chan = cv2.split(lab)

        # Step 1: Contrast Stretch & Gamma LUT on Luminance
        lut = self._build_gamma_lut(params["gamma"])
        l_gamma = cv2.LUT(l_orig, lut)

        # Step 2: Illuminated Edge-Preserving Denoising
        h_val = int(round(params["denoise_h"]))
        if self.denoise_method == "fastnlmeans":
            l_denoised = cv2.fastNlMeansDenoising(l_gamma, None, h=float(h_val), templateWindowSize=7, searchWindowSize=15)
        else:
            d_size = 5 if h_val < 10 else 7
            l_denoised = cv2.bilateralFilter(l_gamma, d=d_size, sigmaColor=h_val * 2.0, sigmaSpace=d_size)

        # Step 3: Controlled CLAHE
        clahe = cv2.createCLAHE(clipLimit=params["clahe_clip"], tileGridSize=(8, 8))
        l_clahe = clahe.apply(l_denoised)
        mix = params.get("clahe_mix", 0.45)
        l_blended = cv2.addWeighted(l_clahe, mix, l_denoised, 1.0 - mix, 0)

        # Step 4: Detail Sharpening (Unsharp Masking)
        blur = cv2.GaussianBlur(l_blended, (0, 0), sigmaX=1.5, sigmaY=1.5)
        mask = cv2.subtract(l_blended, blur)
        l_sharp = cv2.addWeighted(l_blended, 1.0, mask, params["sharpen"], 0)

        # Step 5: Chroma Adjustment
        sat_boost = params.get("sat_boost", 1.15)
        if abs(sat_boost - 1.0) > 0.01:
            a_adj = np.clip((a_chan.astype(np.float32) - 128.0) * sat_boost + 128.0, 0, 255).astype(np.uint8)
            b_adj = np.clip((b_chan.astype(np.float32) - 128.0) * sat_boost + 128.0, 0, 255).astype(np.uint8)
        else:
            a_adj, b_adj = a_chan, b_chan

        merged_lab = cv2.merge([l_sharp, a_adj, b_adj])
        return cv2.cvtColor(merged_lab, cv2.COLOR_LAB2BGR)


# ------------------------------------------------------------------------------
# 2.3 Generative Night-to-Day Style Enhancer
# ------------------------------------------------------------------------------
class GenerativeNightToDayEnhancer:
    """Generative curve and daylight harmonization style synthesizer."""
    def enhance(self, frame_bgr: np.ndarray) -> np.ndarray:
        float_img = frame_bgr.astype(np.float32) / 255.0
        # Multi-scale deep curve iteration
        curve = float_img
        for _ in range(4):
            curve = curve + 0.35 * curve * (1.0 - curve)
        enhanced = np.clip(curve * 255.0, 0, 255).astype(np.uint8)
        
        # Color cast harmonization
        lab = cv2.cvtColor(enhanced, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        l = clahe.apply(l)
        return cv2.cvtColor(cv2.merge([l, a, b]), cv2.COLOR_LAB2BGR)


# ------------------------------------------------------------------------------
# 2.4 Temporal Video Parameter Smoother & Scene Cut Detector
# ------------------------------------------------------------------------------
class TemporalSmoother:
    """Maintains parameter coherence across video frames and detects scene cuts."""
    def __init__(self, alpha: float = 0.88, scene_cut_threshold: float = 0.55):
        self.alpha = float(np.clip(alpha, 0.0, 0.99))
        self.scene_cut_threshold = scene_cut_threshold
        self.previous_params = None
        self.previous_hist = None
        self.scene_cut_count = 0

    def compute_histogram(self, frame_bgr: np.ndarray) -> np.ndarray:
        gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
        hist = cv2.calcHist([gray], [0], None, [32], [0, 256])
        cv2.normalize(hist, hist)
        return hist

    def update(self, current_params: Dict[str, Any], frame_bgr: np.ndarray) -> Dict[str, Any]:
        # Handle swapped argument ordering gracefully
        if isinstance(current_params, np.ndarray) and isinstance(frame_bgr, dict):
            current_params, frame_bgr = frame_bgr, current_params

        current_hist = self.compute_histogram(frame_bgr)
        cut_detected = False
        if self.previous_hist is not None:
            similarity = cv2.compareHist(self.previous_hist, current_hist, cv2.HISTCMP_CORREL)
            cut_detected = bool(similarity < self.scene_cut_threshold)

        if cut_detected:
            self.scene_cut_count += 1
            self.previous_params = None

        if self.previous_params is None:
            smoothed = current_params.copy()
        else:
            smoothed = current_params.copy()
            for key in ("gamma", "denoise_h", "clahe_clip", "sharpen"):
                if key in current_params and key in self.previous_params:
                    smoothed[key] = float(self.alpha * self.previous_params[key] + (1.0 - self.alpha) * current_params[key])

        self.previous_params = smoothed.copy()
        self.previous_hist = current_hist
        return smoothed


## 3. Deep Learning Object Detection (YOLO 11n), SAHI & Micro-Particle Saliency
Implements pure OpenCV DNN inference for YOLO 11n, Sliced Aided Hyper Inference (SAHI) for small/distant targets, morphological micro-particle saliency, and real-time temporal tracking.


In [ ]:
# 80 standard COCO classes
COCO_CLASSES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat",
    "traffic light", "fire hydrant", "stop sign", "parking meter", "bench", "bird", "cat",
    "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "backpack",
    "umbrella", "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard", "sports ball",
    "kite", "baseball bat", "baseball glove", "skateboard", "surfboard", "tennis racket",
    "bottle", "wine glass", "cup", "fork", "knife", "spoon", "bowl", "banana", "apple",
    "sandwich", "orange", "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair",
    "couch", "potted plant", "bed", "dining table", "toilet", "tv", "laptop", "mouse", "remote",
    "keyboard", "cell phone", "microwave", "oven", "toaster", "sink", "refrigerator", "book",
    "clock", "vase", "scissors", "teddy bear", "hair drier", "toothbrush"
]

np.random.seed(42)
CLASS_COLORS = [
    (int(c[0]), int(c[1]), int(c[2]))
    for c in np.random.randint(50, 255, size=(len(COCO_CLASSES), 3))
]


class UnifiedObjectDetector:
    """OpenCV DNN detector supporting YOLO 11n, SAHI high-res tiling, and micro-particle saliency."""
    def __init__(
        self,
        model_path: Optional[str] = None,
        conf_threshold: float = 0.15,
        nms_threshold: float = 0.45,
        target_size: Tuple[int, int] = (640, 640),
        sliced: bool = False,
        slice_size: Tuple[int, int] = (480, 480),
        detect_particles: bool = False,
        particle_prominence: int = 30
    ):
        self.conf_threshold = conf_threshold
        self.nms_threshold = nms_threshold
        self.target_size = target_size
        self.classes = COCO_CLASSES
        self.sliced = sliced
        self.slice_size = slice_size
        self.detect_particles_enabled = detect_particles
        self.particle_prominence = particle_prominence

        # Auto-resolve model
        self.model_path = model_path if model_path else RESOLVED_MODEL_PATH
        if self.model_path is None or not os.path.exists(self.model_path):
            raise FileNotFoundError(f"YOLO 11n ONNX model not found. Expected at models/yolo11n.onnx")

        self.net = cv2.dnn.readNetFromONNX(self.model_path)
        try:
            self.net.setPreferableBackend(cv2.dnn.DNN_BACKEND_OPENCV)
        except Exception:
            pass

    def _letterbox(self, img: np.ndarray, new_shape=(640, 640), color=(114, 114, 114)):
        shape = img.shape[:2]
        r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
        new_unpad = (int(round(shape[1] * r)), int(round(shape[0] * r)))
        dw = (new_shape[1] - new_unpad[0]) / 2.0
        dh = (new_shape[0] - new_unpad[1]) / 2.0
        if shape[::-1] != new_unpad:
            img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
        top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
        left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
        img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)
        return img, r, (dw, dh)

    def _detect_single_frame(self, frame_bgr: np.ndarray) -> List[Dict[str, Any]]:
        orig_h, orig_w = frame_bgr.shape[:2]
        letterboxed, ratio, (dw, dh) = self._letterbox(frame_bgr, self.target_size)
        blob = cv2.dnn.blobFromImage(letterboxed, 1.0 / 255.0, self.target_size, swapRB=True, crop=False)
        self.net.setInput(blob)
        output = self.net.forward()

        preds = output[0].T  # Shape: (8400, 84)
        boxes_candidates = preds[:, :4]
        class_scores = preds[:, 4:]

        max_scores = np.max(class_scores, axis=1)
        class_ids = np.argmax(class_scores, axis=1)
        mask = max_scores >= self.conf_threshold

        filtered_boxes = boxes_candidates[mask]
        filtered_scores = max_scores[mask]
        filtered_ids = class_ids[mask]

        if len(filtered_scores) == 0:
            return []

        boxes_list, confs_list, ids_list = [], [], []
        for i in range(len(filtered_scores)):
            cx, cy, w, h = filtered_boxes[i]
            x1 = max(0, min(orig_w - 1, int(round((cx - w / 2.0 - dw) / ratio))))
            y1 = max(0, min(orig_h - 1, int(round((cy - h / 2.0 - dh) / ratio))))
            x2 = max(0, min(orig_w - 1, int(round((cx + w / 2.0 - dw) / ratio))))
            y2 = max(0, min(orig_h - 1, int(round((cy + h / 2.0 - dh) / ratio))))
            bw, bh = max(1, x2 - x1), max(1, y2 - y1)
            boxes_list.append([x1, y1, bw, bh])
            confs_list.append(float(filtered_scores[i]))
            ids_list.append(int(filtered_ids[i]))

        indices = cv2.dnn.NMSBoxes(boxes_list, confs_list, self.conf_threshold, self.nms_threshold)
        detections = []
        if len(indices) > 0:
            for idx in indices.flatten():
                bx, by, bw, bh = boxes_list[idx]
                cid = ids_list[idx]
                detections.append({
                    "bbox": [bx, by, bw, bh],
                    "box_xyxy": [bx, by, bx + bw, by + bh],
                    "confidence": confs_list[idx],
                    "class_id": cid,
                    "class_name": self.classes[cid] if cid < len(self.classes) else f"class_{cid}"
                })
        return detections

    def detect_particles(self, frame_bgr: np.ndarray) -> List[Dict[str, Any]]:
        """Isolates micro-particles, point-lights, and specks via morphological top-hat."""
        lab = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2LAB)
        l_chan = lab[:, :, 0]
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
        tophat = cv2.morphologyEx(l_chan, cv2.MORPH_TOPHAT, kernel)
        _, thresh = cv2.threshold(tophat, self.particle_prominence, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        particles = []
        for cnt in contours:
            x, y, w, h = cv2.boundingRect(cnt)
            if 1 <= max(w, h) <= 28:
                roi = tophat[y:y+h, x:x+w]
                score = float(np.mean(roi)) / 255.0
                particles.append({
                    "bbox": [x, y, w, h],
                    "box_xyxy": [x, y, x + w, y + h],
                    "confidence": min(0.99, float(score * 2.2)),
                    "class_id": 999,
                    "class_name": "particle"
                })
        return particles

    def detect_sliced(self, frame_bgr: np.ndarray, overlap: float = 0.25) -> List[Dict[str, Any]]:
        """Sliced Aided Hyper Inference (SAHI) tiled inference for small/distant targets."""
        H, W = frame_bgr.shape[:2]
        sw, sh = self.slice_size
        if H <= sh and W <= sw:
            return self._detect_single_frame(frame_bgr)

        step_x = max(100, int(sw * (1.0 - overlap)))
        step_y = max(100, int(sh * (1.0 - overlap)))
        all_dets = list(self._detect_single_frame(frame_bgr))

        for y in range(0, max(1, H - sh + step_y), step_y):
            y_end = min(H, y + sh)
            y_start = max(0, y_end - sh)
            for x in range(0, max(1, W - sw + step_x), step_x):
                x_end = min(W, x + sw)
                x_start = max(0, x_end - sw)
                crop = frame_bgr[y_start:y_end, x_start:x_end]
                for d in self._detect_single_frame(crop):
                    bx, by, bw, bh = d["bbox"]
                    gx, gy = bx + x_start, by + y_start
                    all_dets.append({
                        "bbox": [gx, gy, bw, bh],
                        "box_xyxy": [gx, gy, gx + bw, gy + bh],
                        "confidence": d["confidence"],
                        "class_id": d["class_id"],
                        "class_name": d["class_name"]
                    })

        if not all_dets:
            return []

        boxes = [d["bbox"] for d in all_dets]
        confs = [d["confidence"] for d in all_dets]
        indices = cv2.dnn.NMSBoxes(boxes, confs, self.conf_threshold, self.nms_threshold)
        return [all_dets[i] for i in indices.flatten()] if len(indices) > 0 else []

    def detect(self, frame_bgr: np.ndarray) -> List[Dict[str, Any]]:
        dets = self.detect_sliced(frame_bgr) if self.sliced else self._detect_single_frame(frame_bgr)
        if self.detect_particles_enabled:
            dets.extend(self.detect_particles(frame_bgr))
        return dets

    def draw_detections(self, frame_bgr: np.ndarray, detections: List[Dict[str, Any]], show_labels: bool = True) -> np.ndarray:
        annotated = frame_bgr.copy()
        for det in detections:
            x, y, w, h = det["bbox"]
            cid = det["class_id"]
            cname = det["class_name"]
            conf = det["confidence"]
            track_id = det.get("track_id")
            color = (0, 240, 255) if cname == "particle" else (CLASS_COLORS[(track_id * 37) % len(CLASS_COLORS)] if track_id else CLASS_COLORS[cid % len(CLASS_COLORS)])

            cv2.rectangle(annotated, (x, y), (x + w, y + h), color, 2)
            if show_labels:
                label = f"#{track_id} {cname} {conf:.0%}" if track_id else f"{cname} {conf:.1%}"
                (lw, lh), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.45, 1)
                label_y = max(y, lh + 6)
                cv2.rectangle(annotated, (x, label_y - lh - 4), (x + lw + 6, label_y + baseline - 2), color, cv2.FILLED)
                cv2.putText(annotated, label, (x + 3, label_y - 2), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
        return annotated


# ------------------------------------------------------------------------------
# 3.2 Real-Time Multi-Object Video Tracker
# ------------------------------------------------------------------------------
class VideoTracker:
    """Associates detections across frames using IoU matching & EMA smoothing."""
    def __init__(self, max_missed: int = 4, iou_threshold: float = 0.25, smooth_alpha: float = 0.70):
        self.max_missed = max_missed
        self.iou_threshold = iou_threshold
        self.smooth_alpha = smooth_alpha
        self.next_track_id = 1
        self.tracks = []

    def reset(self):
        self.next_track_id = 1
        self.tracks = []

    @staticmethod
    def _compute_iou(b1, b2) -> float:
        x1, y1, w1, h1 = b1
        x2, y2, w2, h2 = b2
        xi1, yi1 = max(x1, x2), max(y1, y2)
        xi2, yi2 = min(x1 + w1, x2 + w2), min(y1 + h1, y2 + h2)
        inter_area = max(0, xi2 - xi1) * max(0, yi2 - yi1)
        union_area = (w1 * h1) + (w2 * h2) - inter_area
        return inter_area / union_area if union_area > 0 else 0.0

    def update(self, detections: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        updated = []
        unmatched = list(detections)
        for t in self.tracks:
            best_iou, best_idx = 0.0, -1
            for idx, d in enumerate(unmatched):
                if d["class_name"] == t["class_name"]:
                    iou = self._compute_iou(t["bbox"], d["bbox"])
                    if iou > best_iou:
                        best_iou, best_idx = iou, idx
            if best_iou >= self.iou_threshold and best_idx >= 0:
                m = unmatched.pop(best_idx)
                # EMA smooth coordinates
                sb = [
                    int(round(self.smooth_alpha * t["bbox"][k] + (1.0 - self.smooth_alpha) * m["bbox"][k]))
                    for k in range(4)
                ]
                t.update({"bbox": sb, "confidence": m["confidence"], "missed": 0})
                updated.append(t)
            else:
                t["missed"] += 1
                if t["missed"] <= self.max_missed:
                    updated.append(t)

        for d in unmatched:
            d_copy = dict(d)
            d_copy["track_id"] = self.next_track_id
            d_copy["missed"] = 0
            self.next_track_id += 1
            updated.append(d_copy)

        self.tracks = updated
        return [dict(t) for t in self.tracks]


# ------------------------------------------------------------------------------
# 3.3 Quantitative Evaluator & Side-by-Side Canvas Builder
# ------------------------------------------------------------------------------
class DetectionEvaluator:
    @staticmethod
    def compute_metrics(detections: List[Dict[str, Any]]) -> Dict[str, Any]:
        count = len(detections)
        if count == 0:
            return {"total_count": 0, "mean_confidence": 0.0, "class_counts": {}}
        confs = [d["confidence"] for d in detections]
        counts = {}
        for d in detections:
            c = d["class_name"]
            counts[c] = counts.get(c, 0) + 1
        return {"total_count": count, "mean_confidence": float(np.mean(confs)), "class_counts": counts}

    @classmethod
    def compare_detections(cls, raw_dets, enh_dets) -> Dict[str, Any]:
        rm = cls.compute_metrics(raw_dets)
        em = cls.compute_metrics(enh_dets)
        return {
            "raw": rm, "enhanced": em,
            "count_gain": em["total_count"] - rm["total_count"],
            "conf_gain": em["mean_confidence"] - rm["mean_confidence"]
        }

    @staticmethod
    def create_comparison_canvas(raw_ann, enh_ann, metrics, title_raw="Raw Low-Light", title_enh="Enhanced") -> np.ndarray:
        h, w = raw_ann.shape[:2]
        enh_res = cv2.resize(enh_ann, (w, h))
        combined = np.hstack([raw_ann, enh_res])

        banner = np.zeros((70, combined.shape[1], 3), dtype=np.uint8)
        banner[:] = (25, 25, 28)

        rc, rconf = metrics["raw"]["total_count"], metrics["raw"]["mean_confidence"]
        ec, econf = metrics["enhanced"]["total_count"], metrics["enhanced"]["mean_confidence"]
        gain = f"(+{metrics['count_gain']} new)" if metrics["count_gain"] > 0 else ""

        cv2.putText(banner, f"{title_raw} | Detections: {rc} (Avg Conf: {rconf:.1%})", (20, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (180, 180, 220), 2, cv2.LINE_AA)
        cv2.putText(banner, f"{title_enh} | Detections: {ec} {gain} (Avg Conf: {econf:.1%})", (w + 20, 42), cv2.FONT_HERSHEY_SIMPLEX, 0.60, (100, 240, 140), 2, cv2.LINE_AA)
        return np.vstack([banner, combined])


## 4. Unified End-to-End Media Processing Pipelines
Coordinating functions that route images and video streams through illumination recovery and neural detection.


In [ ]:
def process_image_unified(
    image_bgr: np.ndarray,
    enhancer_type: str = "adaptive",
    denoise_method: str = "fastnlmeans",
    detector: Optional[UnifiedObjectDetector] = None,
    conf_threshold: float = 0.12,
    save_comparison: bool = True
) -> Dict[str, Any]:
    """Processes an image through illumination enhancement and object detection."""
    raw_bgr = image_bgr.copy()
    params = select_adaptive_parameters(raw_bgr)

    # 1. Illumination Enhancement
    if enhancer_type == "generative":
        gen = GenerativeNightToDayEnhancer()
        enhanced_bgr = gen.enhance(raw_bgr)
    elif enhancer_type == "both":
        adapt = AdaptiveEnhancer(denoise_method=denoise_method)
        gen = GenerativeNightToDayEnhancer()
        enhanced_bgr = gen.enhance(adapt.enhance(raw_bgr, params))
    else:
        adapt = AdaptiveEnhancer(denoise_method=denoise_method)
        enhanced_bgr = adapt.enhance(raw_bgr, params)

    # 2. Object Detection
    if detector is None:
        detector = UnifiedObjectDetector(conf_threshold=conf_threshold)
    else:
        detector.conf_threshold = conf_threshold

    raw_dets = detector.detect(raw_bgr)
    enh_dets = detector.detect(enhanced_bgr)

    raw_annotated = detector.draw_detections(raw_bgr, raw_dets)
    enh_annotated = detector.draw_detections(enhanced_bgr, enh_dets)

    metrics = DetectionEvaluator.compare_detections(raw_dets, enh_dets)
    canvas = None
    if save_comparison:
        canvas = DetectionEvaluator.create_comparison_canvas(
            raw_annotated, enh_annotated, metrics,
            title_raw="Raw Low-Light Input",
            title_enh=f"Enhanced ({enhancer_type.capitalize()}) + YOLO 11n"
        )

    return {
        "raw_image": raw_bgr,
        "enhanced_image": enhanced_bgr,
        "raw_annotated": raw_annotated,
        "enh_annotated": enh_annotated,
        "comparison_canvas": canvas,
        "params": params,
        "raw_detections": raw_dets,
        "enh_detections": enh_dets,
        "metrics": metrics
    }


def process_video_unified(
    video_path: str,
    output_path: str,
    enhancer_type: str = "adaptive",
    denoise_method: str = "bilateral",
    detector: Optional[UnifiedObjectDetector] = None,
    conf_threshold: float = 0.12,
    max_frames: Optional[int] = None,
    max_width: int = 720,
    progress_callback: Optional[Callable[[int, int], None]] = None
) -> Dict[str, Any]:
    """Processes nocturnal video with temporal smoothing, object detection, and multi-object tracking."""
    cap = cv2.VideoCapture(video_path)
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    orig_w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    orig_h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

    proc_w, proc_h = orig_w, orig_h
    if max_width and orig_w > max_width:
        scale = max_width / float(orig_w)
        proc_w, proc_h = int(round(orig_w * scale)), int(round(orig_h * scale))

    out_w = proc_w * 2
    out_h = proc_h + 70

    os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
    fourcc = cv2.VideoWriter_fourcc(*"avc1")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (out_w, out_h))
    if not writer.isOpened():
        fourcc = cv2.VideoWriter_fourcc(*"mp4v")
        writer = cv2.VideoWriter(output_path, fourcc, fps, (out_w, out_h))

    adapt = AdaptiveEnhancer(denoise_method=denoise_method)
    gen = GenerativeNightToDayEnhancer()
    smoother = TemporalSmoother()
    enh_tracker = VideoTracker()
    raw_tracker = VideoTracker()

    if detector is None:
        detector = UnifiedObjectDetector(conf_threshold=conf_threshold)

    frame_idx = 0
    total_detections = 0
    target_count = max_frames if max_frames else (total_frames if total_frames > 0 else 100)

    try:
        while True:
            if max_frames and frame_idx >= max_frames:
                break
            ret, frame = cap.read()
            if not ret or frame is None:
                break

            frame_idx += 1
            if (frame.shape[1], frame.shape[0]) != (proc_w, proc_h):
                frame = cv2.resize(frame, (proc_w, proc_h), interpolation=cv2.INTER_AREA)

            raw_frame = frame.copy()
            params = select_adaptive_parameters(raw_frame)
            smoothed = smoother.update(params, raw_frame)

            if enhancer_type == "generative":
                enh_frame = gen.enhance(raw_frame)
            elif enhancer_type == "both":
                enh_frame = gen.enhance(adapt.enhance(raw_frame, smoothed))
            else:
                enh_frame = adapt.enhance(raw_frame, smoothed)

            raw_dets = detector.detect(raw_frame)
            enh_dets = detector.detect(enh_frame)
            total_detections += len(enh_dets)

            tracked_raw = raw_tracker.update(raw_dets)
            tracked_enh = enh_tracker.update(enh_dets)

            raw_ann = detector.draw_detections(raw_frame, tracked_raw)
            enh_ann = detector.draw_detections(enh_frame, tracked_enh)

            metrics = DetectionEvaluator.compare_detections(tracked_raw, tracked_enh)
            canvas = DetectionEvaluator.create_comparison_canvas(
                raw_ann, enh_ann, metrics,
                title_raw="Raw Nocturnal Stream",
                title_enh=f"Enhanced + Tracking (YOLO 11n)"
            )
            writer.write(canvas)

            if progress_callback:
                progress_callback(frame_idx, target_count)
    finally:
        cap.release()
        writer.release()

    return {
        "processed_frames": frame_idx,
        "total_detections": total_detections,
        "scene_cuts": smoother.scene_cut_count,
        "output_path": output_path
    }


## 5. 🚀 Interactive Upload & Media Processing Hub
Upload your own low-light images or nocturnal videos directly using the interactive widget below, or choose from the bundled presets.


In [ ]:
# Helper to display HTML5 playable video
def display_video_html5(video_path: str, width: int = 850):
    import base64
    with open(video_path, "rb") as f:
        video_bytes = f.read()
    b64 = base64.b64encode(video_bytes).decode("ascii")
    html_code = f"""
    <div style="background: #111827; padding: 12px; border-radius: 8px; margin-top: 10px;">
        <h4 style="color: #38bdf8; margin: 0 0 10px 0;">🎬 Annotated Video Playback (Raw vs Enhanced + YOLO 11n Tracking)</h4>
        <video width="{width}" controls autoplay loop muted style="border-radius: 6px;">
            <source src="data:video/mp4;base64,{b64}" type="video/mp4">
            Your browser does not support HTML5 video.
        </video>
    </div>
    """
    display(HTML(html_code))


# UI Controls
upload_widget = widgets.FileUpload(
    accept="image/*,video/*,.mp4,.mov,.avi,.jpg,.jpeg,.png,.bmp",
    multiple=False,
    description="Upload File",
    tooltip="Click to upload any dark photo or nocturnal video"
)

preset_dropdown = widgets.Dropdown(
    options=[
        ("(None - Use Uploaded File)", "none"),
        ("ExDark: Pedestrians & Cyclist in Extreme Dark", "data/exdark/images/2015_00013_jpg.rf.1ccbc5a451029f00e87a4e0d4181841b.jpg"),
        ("ExDark: Roadside Vehicle in Extreme Dark", "data/exdark/images/2015_00026_jpg.rf.4ed27f2ff81dd07c2a7b271d8bb85ad3.jpg"),
        ("DICM: High-Dynamic Range Night Scene", "data/dicm/01.jpg"),
        ("LIME: Extreme Low-Light Street", "data/lime/01.bmp"),
        ("Video: Nocturnal Traffic Stream", "data/real_night_video.mp4")
    ],
    value="data/exdark/images/2015_00013_jpg.rf.1ccbc5a451029f00e87a4e0d4181841b.jpg",
    description="Sample Preset:",
    layout=widgets.Layout(width="500px")
)

path_input = widgets.Text(
    value="",
    placeholder="e.g., path/to/my_dark_photo.jpg or video.mp4",
    description="Custom Path:",
    layout=widgets.Layout(width="500px")
)

enhancer_dropdown = widgets.Dropdown(
    options=[("Adaptive (LAB Continuous)", "adaptive"), ("Generative (Night-to-Day)", "generative"), ("Both (Cascaded)", "both")],
    value="adaptive",
    description="Enhancer:"
)

denoise_dropdown = widgets.Dropdown(
    options=[("Fast Bilateral (Real-Time)", "bilateral"), ("FastNLMeans (High Quality)", "fastnlmeans")],
    value="bilateral",
    description="Denoiser:"
)

conf_slider = widgets.FloatSlider(
    value=0.10,
    min=0.01,
    max=0.80,
    step=0.01,
    description="Confidence:",
    tooltip="Lower confidence allows detection of very minute / dark targets"
)

sliced_checkbox = widgets.Checkbox(
    value=True,
    description="Enable Sliced Tiling (SAHI) for Minute Objects",
    tooltip="Scans high-res overlapping tiles to enlarge small/distant targets"
)

particle_checkbox = widgets.Checkbox(
    value=True,
    description="Detect Micro-Particles & Point Lights",
    tooltip="Morphological top-hat detector for micro luminous specks and reflections"
)

run_button = widgets.Button(
    description="🚀 Run Enhancement & Detection",
    button_style="primary",
    icon="play",
    layout=widgets.Layout(width="280px", height="40px")
)

output_area = widgets.Output()


def on_run_clicked(b):
    with output_area:
        clear_output()
        print("⏳ Preparing media for processing...")

        # Resolve media source
        media_bytes = None
        media_path = None
        is_video = False

        if upload_widget.value:
            # ipywidgets v8 format: tuple/list of dicts
            file_info = list(upload_widget.value)[0] if isinstance(upload_widget.value, (list, tuple)) else upload_widget.value[list(upload_widget.value.keys())[0]]
            fname = file_info.get("name", "uploaded_file")
            media_bytes = bytes(file_info.get("content", b""))
            ext = os.path.splitext(fname)[1].lower()
            is_video = ext in [".mp4", ".mov", ".avi", ".mkv"]
            print(f"📁 Loaded uploaded file: '{fname}' ({len(media_bytes) / 1024:.1f} KB)")
        elif path_input.value.strip() and os.path.exists(path_input.value.strip()):
            media_path = path_input.value.strip()
            ext = os.path.splitext(media_path)[1].lower()
            is_video = ext in [".mp4", ".mov", ".avi", ".mkv"]
            print(f"📁 Loaded custom path: '{media_path}'")
        elif preset_dropdown.value != "none":
            media_path = preset_dropdown.value
            ext = os.path.splitext(media_path)[1].lower()
            is_video = ext in [".mp4", ".mov", ".avi", ".mkv"]
            print(f"📁 Loaded preset: '{os.path.basename(media_path)}'")
        else:
            print("❌ Please upload a file, enter a valid path, or select a sample preset.")
            return

        # Initialize detector
        detector = UnifiedObjectDetector(
            conf_threshold=conf_slider.value,
            sliced=sliced_checkbox.value,
            detect_particles=particle_checkbox.value
        )

        # ---------------- IMAGE PROCESSING ----------------
        if not is_video:
            if media_bytes is not None:
                img_bgr = cv2.imdecode(np.frombuffer(media_bytes, np.uint8), cv2.IMREAD_COLOR)
            else:
                img_bgr = cv2.imread(media_path)

            if img_bgr is None:
                print("❌ Could not decode input image.")
                return

            t0 = time.time()
            res = process_image_unified(
                img_bgr,
                enhancer_type=enhancer_dropdown.value,
                denoise_method=denoise_dropdown.value,
                detector=detector,
                conf_threshold=conf_slider.value,
                save_comparison=True
            )
            elapsed = time.time() - t0

            m = res["metrics"]
            p = res["params"]
            raw_cnt = m["raw"]["total_count"]
            enh_cnt = m["enhanced"]["total_count"]
            gain = m["count_gain"]

            print("=" * 65)
            print("📊 DETECTION & ENHANCEMENT PERFORMANCE SUMMARY")
            print("=" * 65)
            print(f"• Processing Time:         {elapsed*1000:.1f} ms")
            print(f"• Estimated Noise (MAD):   σ = {p['noise_sigma']:.2f}")
            print(f"• Dynamic LAB Gamma:       γ = {p['gamma']:.3f}")
            print(f"• Raw Dark Detections:     {raw_cnt} objects (Avg Conf: {m['raw']['mean_confidence']:.1%})")
            print(f"• Enhanced Detections:     {enh_cnt} objects (Avg Conf: {m['enhanced']['mean_confidence']:.1%})")
            print(f"• Detection Recall Boost:  +{gain} objects detected after enhancement!")
            print("=" * 65)

            # Display comparative canvas
            canvas_rgb = cv2.cvtColor(res["comparison_canvas"], cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(15, 8))
            plt.imshow(canvas_rgb)
            plt.axis("off")
            plt.title("Side-by-Side Comparison: Raw Low-Light vs Enhanced with YOLO 11n & Minute Target Saliency", fontsize=12, pad=12)
            plt.show()

            # Display detected objects breakdown
            dets = res["enh_detections"]
            if dets:
                print(f"\n📋 Detected Objects Breakdown ({len(dets)} total instances):")
                class_counts = {}
                for d in dets:
                    c = d["class_name"]
                    class_counts[c] = class_counts.get(c, 0) + 1
                for cname, cnt in class_counts.items():
                    print(f"  • {cname}: {cnt} instances")
            else:
                print("ℹ️ No objects found above confidence threshold.")

        # ---------------- VIDEO PROCESSING ----------------
        else:
            if media_bytes is not None:
                tfile = tempfile.NamedTemporaryFile(delete=False, suffix=".mp4")
                tfile.write(media_bytes)
                input_vid = tfile.name
            else:
                input_vid = media_path

            out_vid = os.path.join("outputs", "unified_processed_video.mp4")
            prog_bar = widgets.IntProgress(value=0, min=0, max=100, description="Processing:", bar_style="info")
            display(prog_bar)

            def p_cb(cur, total):
                prog_bar.value = min(100, int((cur / max(1, total)) * 100))

            t0 = time.time()
            vres = process_video_unified(
                input_vid,
                out_vid,
                enhancer_type=enhancer_dropdown.value,
                denoise_method=denoise_dropdown.value,
                detector=detector,
                conf_threshold=conf_slider.value,
                max_width=640,
                progress_callback=p_cb
            )
            vtime = time.time() - t0

            print(f"✅ Video processing complete: {vres['processed_frames']} frames in {vtime:.1f}s ({vres['processed_frames'] / max(0.1, vtime):.1f} FPS)!")
            print(f"🎬 Total objects tracked across stream: {vres['total_detections']}")
            display_video_html5(out_vid)


run_button.on_click(on_run_clicked)

# Render UI layout
ui_layout = widgets.VBox([
    widgets.HTML("<h3 style='color: #0284c7; margin-bottom: 4px;'>📁 Media Input Source (Select One)</h3>"),
    widgets.HBox([upload_widget, preset_dropdown]),
    path_input,
    widgets.HTML("<hr style='margin: 8px 0;'>"),
    widgets.HTML("<h3 style='color: #0284c7; margin-bottom: 4px;'>⚙️ Algorithmic Enhancement & Target Detection Controls</h3>"),
    widgets.HBox([enhancer_dropdown, denoise_dropdown]),
    conf_slider,
    widgets.HBox([sliced_checkbox, particle_checkbox]),
    widgets.HTML("<br>"),
    run_button,
    widgets.HTML("<hr style='margin: 12px 0;'>"),
    output_area
])

display(ui_layout)


## 6. Multi-Dataset Quantitative Evaluation (ExDark Benchmark)
Runs automated validation across real low-light nocturnal samples to measure quantitative recall and confidence improvements.


In [ ]:
# Run quick verification across ExDark test images
test_samples = glob.glob("data/exdark/images/*.jpg")[:4]

if test_samples:
    print(f"Running automated validation on {len(test_samples)} ExDark nocturnal test images...")
    detector = UnifiedObjectDetector(conf_threshold=0.10, sliced=True, detect_particles=False)
    
    total_raw, total_enh = 0, 0
    for p in test_samples:
        img = cv2.imread(p)
        if img is None: continue
        r = process_image_unified(img, detector=detector, conf_threshold=0.10, save_comparison=False)
        total_raw += r["metrics"]["raw"]["total_count"]
        total_enh += r["metrics"]["enhanced"]["total_count"]
        
    print("-" * 55)
    print(f"• Total Raw Dark Detections:      {total_raw} objects")
    print(f"• Total Enhanced Detections:      {total_enh} objects")
    print(f"• Overall Recall Gain:            +{total_enh - total_raw} objects (+{((total_enh - total_raw) / max(1, total_raw))*100:.1f}%)")
    print("-" * 55)
else:
    print("No ExDark sample images found at data/exdark/images/.")
